# Dock 02

## Configuration

In [87]:
#pull in local configuration
%run config.py
!cat config.py

# config.py
# This file wants to be listed in .gitignore

GNINA_LOC = "/home/dwaine/octoberproject/gnina"
GNINA_PARAMETER = " --no_gpu"


In [11]:
ligdictjson = "preprocessed/ligand.json"
prodictjson = "preprocessed/protein.json"

prodir = "preprocessed/proteinprep/"
dockeddir = "preprocessed/ligandprep/"
idealdir = "preprocessed/Ideal_Ligand/"
idealdir = "preprocessed/minimized_ideal_ligand/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
results = resultsdir + "results.csv"

In [9]:
sourceproteinprocesseddir = "../proteinprep01/chemfiles/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/Ideal_Ligand/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/minimized_ideal_ligand/"
localprocesseddir = "preprocessed/"

In [12]:
experimentdescription = "Dock 02 is a upgraded version of Dock 01 that utilizes a proper dataframe for tracking gnina results."

In [13]:
proteindict = {}
dockedliganddict = {}
idealliganddict = {}
gninaoptdict = {}
prepdict = {}

In [5]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

In [10]:
#copy preprocessed proteins and docked ligand files
!cp -R {sourceproteinprocesseddir}* {localprocesseddir}
!cp -R {sourceligandidealdir} {localprocesseddir}
!ls -al {localprocesseddir}

total 36
drwxr-xr-x 7 dwaine dwaine 4096 Mar 25 09:33 .
drwxr-xr-x 5 dwaine dwaine 4096 Mar 25 09:32 ..
drwxr-xr-x 2 dwaine dwaine 4096 Mar 24 10:14 Ideal_Ligand
-rw-r--r-- 1 dwaine dwaine 2953 Mar 25 09:33 ligand.json
drwxr-xr-x 2 dwaine dwaine 4096 Mar 24 10:14 ligandprep
drwxr-xr-x 2 dwaine dwaine 4096 Mar 25 09:33 minimized_ideal_ligand
-rw-r--r-- 1 dwaine dwaine 3850 Mar 25 09:33 protein.json
drwxr-xr-x 2 dwaine dwaine 4096 Mar 24 10:14 proteindownload
drwxr-xr-x 2 dwaine dwaine 4096 Mar 24 10:14 proteinprep


## Populate docked ligand dictionary

In [14]:
dockedliganddict = {}

In [15]:
import json

# Read from text file
with open(ligdictjson, "r") as f:
    dockedliganddict = json.load(f)

#Which ligands are available?
lines = reportdict(dockedliganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id               source    prep          localfilename        
EST_redock_1ERE  1ERE.pdb  rdkit_save_H  EST_redock_1ERE_A.sdf
EST_redock_1GWR  1GWR.pdb  rdkit_save_H  EST_redock_1GWR_A.sdf
EST_redock_3UUD  3UUD.pdb  rdkit_save_H  EST_redock_3UUD_A.sdf
EST_redock_6CBZ  6CBZ.pdb  rdkit_save_H  EST_redock_6CBZ_A.sdf
DES_redock_3ERD  3ERD.pdb  rdkit_save_H  DES_redock_3ERD_A.sdf
DES_redock_4ZN7  4ZN7.pdb  rdkit_save_H  DES_redock_4ZN7_A.sdf
27M_redock_4MGC  4MGC.pdb  rdkit_save_H  27M_redock_4MGC_A.sdf
27J_redock_4MG8  4MG8.pdb  rdkit_save_H  27J_redock_4MG8_A.sdf
36J_redock_4TUZ  4TUZ.pdb  rdkit_save_H  36J_redock_4TUZ_A.sdf
2OH_redock_3UU7  3UU7.pdb  rdkit_save_H  2OH_redock_3UU7_A.sdf
27K_redock_4MG9  4MG9.pdb  rdkit_save_H  27K_redock_4MG9_A.sdf
27L_redock_4MGA  4MGA.pdb  rdkit_save_H  27L_redock_4MGA_A.sdf
EST_redock_1G50  1G50.pdb  rdkit_save_H  EST_redock_1G50_A.sdf


## Populate protein dictionary

In [16]:
import json

# Read from text file
with open(prodictjson, "r") as f:
    proteindict = json.load(f)    

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id    name  prep          localfilename_fixed
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb   
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb   
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb   
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb   
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb   
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb   
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb   
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb   
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb   
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb   
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb   
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb   
1G50        PDBfixfunc74  1G50_A_fixed.pdb   


In [17]:
# Add docked ligand id into protein dictionary. Next iteration this will happen in the 'pre process protein' code.

for (k1, v1), (k2, v2) in zip(proteindict.items(), dockedliganddict.items()):
    #print(f"Key: {k1}, Dict1: {v1}, Dict2: {v2}")
    v1['dockedid'] = v2['id']

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed","dockedid"])
print("\n".join(lines))

id    name  prep          localfilename_fixed  dockedid       
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb     EST_redock_1ERE
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb     EST_redock_1GWR
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb     EST_redock_3UUD
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb     EST_redock_6CBZ
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb     DES_redock_3ERD
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb     DES_redock_4ZN7
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb     27M_redock_4MGC
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb     27J_redock_4MG8
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb     36J_redock_4TUZ
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb     2OH_redock_3UU7
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb     27K_redock_4MG9
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb     27L_redock_4MGA
1G50        PDBfixfunc74  1G50_A_fixed.pdb     EST_redock_1G50


## Populate ideal ligand dictionary

In [18]:
idealliganddict = {}

In [19]:
#minimized versions of ideal ligands
idealliganddict["27J_min"] = {'id':"27J_min", 'prep': "obabel-mmff94", 'localfilename': "27J_min.sdf"}
idealliganddict["27K_min"] = {'id':"27K_min", 'prep': "obabel-mmff94", 'localfilename': "27K_min.sdf"}
idealliganddict["27L_min"] = {'id':"27L_min", 'prep': "obabel-mmff94", 'localfilename': "27L_min.sdf"}

idealliganddict["27M_min"] = {'id':"27M_min", 'prep': "obabel-mmff94", 'localfilename': "27M_min.sdf"}
idealliganddict["2OH_min"] = {'id':"2OH_min", 'prep': "obabel-mmff94", 'localfilename': "2OH_min.sdf"}
idealliganddict["36J_min"] = {'id':"36J_min", 'prep': "obabel-mmff94", 'localfilename': "36J_min.sdf"}

idealliganddict["Caffeine_min"] = {'id':"Caffeine_min", 'prep': "obabel-mmff94", 'localfilename': "Caffeine_min.sdf"}
idealliganddict["DES_min"] = {'id':"DES_min", 'prep': "obabel-mmff94", 'localfilename': "DES_min.sdf"}
idealliganddict["EE2_min"] = {'id':"EE2_min", 'prep': "obabel-mmff94", 'localfilename': "EE2_min.sdf"}

idealliganddict["EST_min"] = {'id':"EST_min", 'prep': "obabel-mmff94", 'localfilename': "EST_min.sdf"}
idealliganddict["Melatonin_min"] = {'id':"Melatonin_min", 'prep': "obabel-mmff94", 'localfilename': "Melatonin_min.sdf"}
idealliganddict["Testosterone_min"] = {'id':"Testosterone_min", 'prep': "obabel-mmff94", 'localfilename': "Testosterone_min.sdf"}

In [ ]:
#Ideal ligands
idealliganddict["27J"] = {'id':"27J", 'prep': "obabel-mmff94", 'localfilename': "27J.sdf"}
idealliganddict["27K"] = {'id':"27K", 'prep': "obabel-mmff94", 'localfilename': "27K.sdf"}
idealliganddict["27L"] = {'id':"27L", 'prep': "obabel-mmff94", 'localfilename': "27L.sdf"}

idealliganddict["27M"] = {'id':"27M", 'prep': "obabel-mmff94", 'localfilename': "27M.sdf"}
idealliganddict["2OH"] = {'id':"2OH", 'prep': "obabel-mmff94", 'localfilename': "2OH.sdf"}
idealliganddict["36J"] = {'id':"36J", 'prep': "obabel-mmff94", 'localfilename': "36J.sdf"}

idealliganddict["Caffeine"] = {'id':"Caffeine", 'prep': "obabel-mmff94", 'localfilename': "Caffeine.sdf"}
idealliganddict["DES"] = {'id':"DES", 'prep': "obabel-mmff94", 'localfilename': "DES.sdf"}
idealliganddict["EE2"] = {'id':"EE2", 'prep': "obabel-mmff94", 'localfilename': "EE2.sdf"}

idealliganddict["EST"] = {'id':"EST", 'prep': "obabel-mmff94", 'localfilename': "EST.sdf"}
idealliganddict["Melatonin"] = {'id':"Melatonin", 'prep': "obabel-mmff94", 'localfilename': "Melatonin.sdf"}
idealliganddict["Testosterone"] = {'id':"Testosterone", 'prep': "obabel-mmff94", 'localfilename': "Testosterone.sdf"}

In [20]:
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))

id                prep           localfilename       
27J_min           obabel-mmff94  27J_min.sdf         
27K_min           obabel-mmff94  27K_min.sdf         
27L_min           obabel-mmff94  27L_min.sdf         
27M_min           obabel-mmff94  27M_min.sdf         
2OH_min           obabel-mmff94  2OH_min.sdf         
36J_min           obabel-mmff94  36J_min.sdf         
Caffeine_min      obabel-mmff94  Caffeine_min.sdf    
DES_min           obabel-mmff94  DES_min.sdf         
EE2_min           obabel-mmff94  EE2_min.sdf         
EST_min           obabel-mmff94  EST_min.sdf         
Melatonin_min     obabel-mmff94  Melatonin_min.sdf   
Testosterone_min  obabel-mmff94  Testosterone_min.sdf


## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodir + value['localfilename_fixed'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

In [ ]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))


## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## Define Gnina and dataframe functions

In [89]:
#Define the functions that call Gnina, parse the results files, and write those results to the dataframe.
from rdkit import Chem
number_of_modes = 1 #Report how many modes from each Gnina run?

def executegnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = idealliganddict[ligandid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = idealdir + ligand["localfilename"]
    b = dockeddir + box["localfilename"]
    
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  
    #!"{GNINA_LOC}" -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore "{GNINA_PARAMETER}"

    gnina_cmd = f"{GNINA_LOC} -r {p} -l {l} --autobox_ligand {b} -o {docked} --log {log}"
    gnina_cmd += "--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore"
    gnina_cmd += f"{GNINA_PARAMETER}"
    print(gnina_cmd)
    ! $gnina_cmd
    
    #Compute RMSD of two docked ligand files and save results to rmsdlog file.
    !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"
    
def getdockedresultdf(docked):
    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
        })
    df = pd.DataFrame(rows)
    return(df)

def writeresulttodf(pro,lig,box):
    resultsdf = getdockedresultdf(docked)
    #for index, row in resultsdf.iterrows():
    for index, row in resultsdf.head(number_of_modes).iterrows():
        #write to the resultsdf here.
        temp = [pro, lig, box, "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD'])]
        outputdf.loc[len(outputdf)] = temp


## Run the docking simulations

In [61]:
# Establish new empty dataframe
import pandas as pd
outputdf = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])

In [90]:
#Test block. Dock just one pair.
pkey = "1ERE"
ikey = "EE2_min"
dockedid = "EST_redock_1ERE"
executegnina(pkey, ikey, dockedid)
writeresulttodf(pkey, ikey, dockedid) 

/home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/EE2_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/EE2_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --n

[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:52:46] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [72]:
# Iterate though protein dictionary and ligand dictionary and perform Gnina docking for every combo.
from itertools import islice

pro = proteindict
pro = dict(islice(proteindict.items(), 3))
ide = idealliganddict
ide = dict(islice(idealliganddict.items(), 3, 6))

for pkey, pvalue in pro.items():
    for ikey, ivalue in ide.items():
        print(f"{pkey} meets {ikey} at {pvalue['dockedid']}")
        executegnina(pkey, ikey, pvalue['dockedid'])
        writeresulttodf(pkey,ikey,pvalue['dockedid'])   


1ERE meets 27M_min at EST_redock_1ERE
              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/27M_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
8571 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   C

[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:17:21] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/2OH_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
6623 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | po

[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:18:13] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/36J_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
5284645 | pose 0 | initial pose not within box
5284645 | pose 0 | ligand outside box

mode |  affinity  |  intramol  |    CNN     | 

...

[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/3UUD_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/2OH_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_3UUD_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
6623 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | po

[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:23:57] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/3UUD_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/36J_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_3UUD_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
5284645 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) |

[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:24:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [91]:
# Preview results
print(outputdf)

   protein ideal_ligand    native_ligand CNN_pose  CNN_VS    RMSD
0     1ERE      27J_min  EST_redock_1ERE   0.6478  4.6592     inf
1     1ERE      27K_min  EST_redock_1ERE   0.8967  4.8425     inf
2     1ERE      27L_min  EST_redock_1ERE   0.9506  6.8079     inf
3     1GWR      27J_min  EST_redock_1GWR   0.6222  4.0833     inf
4     1GWR      27K_min  EST_redock_1GWR   0.9136  4.9213     inf
5     1GWR      27L_min  EST_redock_1GWR   0.9051  6.1798     inf
6     3UUD      27J_min  EST_redock_3UUD   0.6729  4.5454     inf
7     3UUD      27K_min  EST_redock_3UUD   0.9189  5.2819     inf
8     3UUD      27L_min  EST_redock_3UUD   0.9499  6.7288     inf
9     1ERE      27M_min  EST_redock_1ERE   0.9499  6.7288     inf
10    1ERE      2OH_min  EST_redock_1ERE   0.9499  6.7288     inf
11    1ERE      36J_min  EST_redock_1ERE   0.9499  6.7288     inf
12    1GWR      27M_min  EST_redock_1GWR   0.9499  6.7288     inf
13    1GWR      2OH_min  EST_redock_1GWR   0.9499  6.7288     inf
14    1GWR

## Order, rearrance, output as .csv

In [74]:
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])

In [75]:
# Order by ligand, then CNN_pose desc. Change order of columns.

for group_name, group_df in outputdf.groupby("ideal_ligand"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]]

print(outputdfsorted)

   ideal_ligand protein    native_ligand CNN_pose  CNN_VS RMSD
0       27J_min    3UUD  EST_redock_3UUD   0.6729  4.5454  inf
1       27J_min    1ERE  EST_redock_1ERE   0.6478  4.6592  inf
2       27J_min    1GWR  EST_redock_1GWR   0.6222  4.0833  inf
3       27K_min    3UUD  EST_redock_3UUD   0.9189  5.2819  inf
4       27K_min    1GWR  EST_redock_1GWR   0.9136  4.9213  inf
5       27K_min    1ERE  EST_redock_1ERE   0.8967  4.8425  inf
6       27L_min    1ERE  EST_redock_1ERE   0.9506  6.8079  inf
7       27L_min    3UUD  EST_redock_3UUD   0.9499  6.7288  inf
8       27L_min    1GWR  EST_redock_1GWR   0.9051  6.1798  inf
9       27M_min    1ERE  EST_redock_1ERE   0.9499  6.7288  inf
10      27M_min    1GWR  EST_redock_1GWR   0.9499  6.7288  inf
11      27M_min    3UUD  EST_redock_3UUD   0.9499  6.7288  inf
12      27M_min    1ERE  EST_redock_1ERE   0.9499  6.7288  inf
13      27M_min    1GWR  EST_redock_1GWR   0.9499  6.7288  inf
14      27M_min    3UUD  EST_redock_3UUD   0.9499  6.72

In [44]:
# Write to .csv
import csv
outputdfsorted.to_csv(results, mode='w', index=False, header=True)